# Evaluation 06 — Export verified run tables for LaTeX

Reads Evaluation 04 results, **never fabricates entries**. Produces a standalone `tabular` to copy into the paper after methodological and data checks. Do not overwrite the manuscript's blank tables automatically.

In [ ]:
# Common Kaggle bootstrap: either attach DermSynReal source as Kaggle Input,
# or set CLONE_REPO_IF_MISSING=True and enable Kaggle Internet.
from pathlib import Path
import subprocess,sys,os,json
REPO_ROOT=Path("/kaggle/input/EDIT_REPO_DATASET/DermSynReal")
CLONE_REPO_IF_MISSING=True
EXPECTED_GIT_SHA=""  # Recommended: set a full pinned SHA for reproducible final runs.
if not (REPO_ROOT/"src/dermsynreal/evaluation.py").is_file():
    if not CLONE_REPO_IF_MISSING:
        raise FileNotFoundError("Attach the repository and change REPO_ROOT, or enable Git clone.")
    REPO_ROOT=Path("/kaggle/working/DermSynReal")
    if not (REPO_ROOT/".git").is_dir():
        subprocess.check_call(["git","clone","https://github.com/shanmukhdatta/DermSynReal.git",str(REPO_ROOT)])
if EXPECTED_GIT_SHA:
    if not (REPO_ROOT/".git").is_dir():
        raise ValueError("For pinned SHA verification attach a Git checkout or enable clone")
    actual=subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"],text=True).strip()
    if actual!=EXPECTED_GIT_SHA:
        raise ValueError(f"Repository SHA {actual} differs from pinned SHA {EXPECTED_GIT_SHA}")
elif (REPO_ROOT/".git").is_dir():
    print("Current Git SHA:",subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"],text=True).strip())
    print("WARNING: pin EXPECTED_GIT_SHA for final research outputs")
sys.path.insert(0,str(REPO_ROOT/"src"))
print("DermSynReal source:",REPO_ROOT)


In [ ]:
REPORT_DIR="/kaggle/input/EDIT_EVAL_04_RESULTS"
OUT_DIR="/kaggle/working/dermsynreal_eval_06"
BUILD_TABLES=False

In [ ]:
if BUILD_TABLES:
    import csv
    source=Path(REPORT_DIR)/"run_summary.csv"
    assert source.is_file(),"Run Evaluation 04 first"
    with source.open(newline="",encoding="utf-8") as f:rows=list(csv.DictReader(f))
    assert rows,"No empirical data to export"
    def latex_escape(s):
        for a,b in [("\\",r"\textbackslash{}"),("_",r"\_"),("%",r"\%"),("&",r"\&"),("#",r"\#")]:
            s=s.replace(a,b)
        return s
    table=[r"\begin{tabular}{llrrr}",r"\hline",
           r"Arm & Backbone / seed & Real macro-F1 & Real balanced accuracy & Real accuracy \\",
           r"\hline"]
    for r in sorted(rows,key=lambda x:(x["backbone"],x["condition"],x["seed"])):
        table.append("{} & {} & {:.4f} & {:.4f} & {:.4f} \\\\".format(
            latex_escape(r["condition"]),latex_escape(r["backbone"]+" / "+r["seed"]),
            float(r["real_test_macro_f1"]),float(r["real_test_balanced_accuracy"]),
            float(r["real_test_accuracy"])))
    table += [r"\hline",r"\end{tabular}"]
    dest=Path(OUT_DIR);dest.mkdir(parents=True,exist_ok=True)
    (dest/"observed_primary_results.tex").write_text("\n".join(table)+"\n",encoding="utf-8")
    print("\n".join(table))
else:
    print("No LaTeX results generated. Requires real verified run_summary.csv from Evaluation 04.")


Fill the main manuscript results only after evidence review, source-class checks, group/label audits, and all planned experiments. Report missing values as unavailable, not as zeros.